
# Prior sensitivity: uniform vs Gaussian on metallicity

Metallicity is often poorly constrained by optical photometry. This recipe
compares two fits: one with a Uniform prior on met_logzsol (weak constraint)
and one with a Gaussian prior (informative from external data). We show how
the posterior distribution changes, and that the same mock data leads to
different inferences depending on the prior.


In [ ]:
import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

ssp = tengri.load_ssp()
bands = ["sdss_u", "sdss_g", "sdss_r", "sdss_i", "sdss_z"]
obs = tengri.Observation(photometry=tengri.Photometry.from_names(bands))

# Generate mock data at subsolar metallicity
key = jax.random.PRNGKey(42)
true_params = {
    "sfh_tsnorm_log_peak_sfr": 0.8,
    "sfh_tsnorm_peak_lbt_gyr": 2.0,
    "sfh_tsnorm_width_gyr": 1.5,
    "sfh_tsnorm_skew": 0.1,
    "sfh_tsnorm_trunc": 5.0,
    "met_logzsol": -0.5,
    "dust_tau_bc": 0.1,
    "dust_tau_diff": 0.2,
    "dust_slope": -0.7,
    "redshift": 0.1,
}

model_template = tengri.SEDModel.build(
    ssp,
    observation=obs,
    sfh={
        "type": "tsnorm",
        "*": tengri.FIXED,
        **{k: v for k, v in true_params.items() if k.startswith("sfh_")},
    },
    dust={
        "type": "two_component",
        "*": tengri.FIXED,
        **{k: v for k, v in true_params.items() if k.startswith("dust_")},
    },
    redshift=tengri.Fixed(true_params["redshift"]),
)
mock = model_template.mock(true_params, snr=20.0, key=key)

# Fit 1: Uniform prior on metallicity
model_uniform = tengri.SEDModel.build(
    ssp,
    observation=obs,
    sfh={"type": "tsnorm", "*": tengri.FREE},
    dust={"type": "two_component", "*": tengri.FREE},
    redshift=tengri.Fixed(0.1),
)
forward_u = tengri.ForwardModel.build(sed=model_uniform, observation=obs)
post_u = forward_u.fit(
    mock.flux_obs, mock.noise, method="vi_native", n_iter=8, n_samples=3, verbose=False
)

# Fit 2: Gaussian prior on metallicity (informative)
# NOTE: To use Gaussian priors, build with Parameters API then construct SEDModel
spec_gaussian = tengri.Parameters(
    sfh_tsnorm_log_peak_sfr=tengri.Uniform(-1.0, 2.5),
    sfh_tsnorm_peak_lbt_gyr=tengri.Uniform(0.5, 12.0),
    sfh_tsnorm_width_gyr=tengri.Uniform(0.3, 5.0),
    sfh_tsnorm_skew=tengri.Uniform(-3.0, 3.0),
    sfh_tsnorm_trunc=tengri.Uniform(1.0, 10.0),
    met_logzsol=tengri.Gaussian(mu=0.0, sigma=0.3),
    dust_tau_diff=tengri.Uniform(0.0, 1.5),
    dust_slope=tengri.Fixed(-0.7),
    redshift=tengri.Fixed(0.1),
    mean_sfh_type="tsnorm",
)
model_gaussian = tengri.SEDModel(spec_gaussian, ssp, observation=obs)
forward_g = tengri.ForwardModel.build(sed=model_gaussian, observation=obs)
post_g = forward_g.fit(
    mock.flux_obs, mock.noise, method="vi_native", n_iter=8, n_samples=3, verbose=False
)

# Plot: Posterior histograms with prior overlays
met_u = np.array(post_u.samples["met_logzsol"])
met_g = np.array(post_g.samples["met_logzsol"])

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.5))

# Uniform prior
axes[0].hist(met_u, bins=20, alpha=0.5, color="C0", density=True, label="Posterior")
z_vals = np.linspace(-2.0, 0.2, 100)
axes[0].plot(z_vals, np.ones_like(z_vals) / 2.2, "k--", lw=1.2, label="Prior")
axes[0].axvline(-0.5, color="red", ls=":", lw=1.2, label="True value")
axes[0].set_xlabel(r"$\log_{10}(Z/Z_\odot)$")
axes[0].set_ylabel("Probability density")
axes[0].legend(frameon=False, fontsize=8)
ax_text = axes[0].text(
    0.05, 0.95, "Uniform prior", transform=axes[0].transAxes, fontsize=9, verticalalignment="top"
)

# Gaussian prior
axes[1].hist(met_g, bins=20, alpha=0.5, color="C3", density=True, label="Posterior")
axes[1].plot(
    z_vals,
    np.exp(-0.5 * (z_vals / 0.3) ** 2) / (0.3 * np.sqrt(2 * np.pi)),
    "k--",
    lw=1.2,
    label="Prior",
)
axes[1].axvline(-0.5, color="red", ls=":", lw=1.2, label="True value")
axes[1].set_xlabel(r"$\log_{10}(Z/Z_\odot)$")
axes[1].set_ylabel("Probability density")
axes[1].legend(frameon=False, fontsize=8)
ax_text = axes[1].text(
    0.05,
    0.95,
    "Gaussian N(0, 0.3)",
    transform=axes[1].transAxes,
    fontsize=9,
    verticalalignment="top",
)

fig.tight_layout()
fig.savefig("plot_recipe_compare_priors.png", dpi=150, bbox_inches="tight")